In [2]:
import pandas as pd

In [4]:
dfc = pd.read_csv('D:\Emi Predict\Data Sets\emi_prediction_cleanafterEDA.csv')

<>:1: SyntaxWarning: invalid escape sequence '\E'
<>:1: SyntaxWarning: invalid escape sequence '\E'
C:\Users\Windows 10\AppData\Local\Temp\ipykernel_12552\1469425064.py:1: SyntaxWarning: invalid escape sequence '\E'
  dfc = pd.read_csv('D:\Emi Predict\Data Sets\emi_prediction_cleanafterEDA.csv')


In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [13]:
dfc['debt_to_income'] = (
    (dfc['current_emi_amount'] + dfc['existing_loans']) /
    dfc['monthly_salary']
)

In [15]:
dfc['total_expenses'] = (
    dfc['monthly_rent'] +
    dfc['school_fees'] +
    dfc['college_fees'] +
    dfc['travel_expenses'] +
    dfc['groceries_utilities'] +
    dfc['other_monthly_expenses']
)

dfc['expense_to_income'] = dfc['total_expenses'] / dfc['monthly_salary']

In [16]:
dfc['emi_affordability'] = dfc['max_monthly_emi'] / dfc['monthly_salary']

In [17]:
dfc['savings_ratio'] = (
    (dfc['bank_balance'] + dfc['emergency_fund']) /
    dfc['monthly_salary']
)

In [18]:
dfc['credit_risk_score'] = dfc['credit_score'] / 850

In [19]:
dfc['employment_stability'] = dfc['years_of_employment'] / dfc['age']

In [20]:
dfc['financial_stability'] = (
    dfc['bank_balance'] +
    dfc['emergency_fund']
) / dfc['total_expenses']

In [21]:
dfc['salary_credit_interaction'] = (
    dfc['monthly_salary'] * dfc['credit_score']
)

In [22]:
dfc['loan_income_ratio'] = (
    dfc['requested_amount'] /
    dfc['monthly_salary']
)

In [23]:
dfc['emi_burden'] = (
    dfc['current_emi_amount'] /
    dfc['monthly_salary']
)

In [6]:
target_encoder = LabelEncoder()

dfc['emi_eligibility'] = target_encoder.fit_transform(dfc['emi_eligibility'])

In [8]:
dfc['emi_eligibility'].unique()

array([0, 2, 1])

In [7]:
from sklearn.preprocessing import LabelEncoder

# Label Encoding — Target column
le = LabelEncoder()
dfc['emi_eligibility'] = le.fit_transform(dfc['emi_eligibility']) #not eligible -0 , 1 - high, 2 - eligible

# Verify
print(dfc['emi_eligibility'].value_counts())
print(f"\nClasses: {le.classes_}")
print("\nTarget Label Encoding Done! ✅")

emi_eligibility
0    312868
2     74444
1     17488
Name: count, dtype: int64

Classes: [0 1 2]

Target Label Encoding Done! ✅


In [9]:
# Check skewness of all numerical columns
numerical_cols = dfc.select_dtypes(include=['int64', 'float64']).columns
# Remove target and encoded columns
cols_to_scale = [col for col in numerical_cols if col not in ['emi_eligibility'] and dfc[col].nunique() > 2]

print("Skewness of numerical columns:")
print(dfc[cols_to_scale].skew().sort_values(ascending=False))

Skewness of numerical columns:
monthly_salary            4.831152
bank_balance              1.422707
college_fees              1.040627
requested_tenure          0.918680
age                       0.652411
school_fees               0.584345
current_emi_amount        0.415194
monthly_rent              0.378475
education                 0.298449
house_type                0.091839
family_size               0.022211
dependents                0.022211
emi_scenario             -0.000145
employment_type          -0.075761
groceries_utilities      -0.181008
travel_expenses          -0.183910
other_monthly_expenses   -0.213300
requested_amount         -0.273279
max_monthly_emi          -0.334171
company_type             -0.408423
years_of_employment      -0.470603
emergency_fund           -0.656365
credit_score             -1.097378
dtype: float64


In [10]:
print(dfc[numerical_cols].skew().sort_values(ascending=False))

monthly_salary            4.831152
emi_eligibility           1.454976
bank_balance              1.422707
marital_status            1.220566
college_fees              1.040627
requested_tenure          0.918680
age                       0.652411
school_fees               0.584345
current_emi_amount        0.415194
existing_loans            0.411897
monthly_rent              0.378475
education                 0.298449
house_type                0.091839
family_size               0.022211
dependents                0.022211
emi_scenario             -0.000145
employment_type          -0.075761
groceries_utilities      -0.181008
travel_expenses          -0.183910
other_monthly_expenses   -0.213300
requested_amount         -0.273279
max_monthly_emi          -0.334171
company_type             -0.408423
gender                   -0.408985
years_of_employment      -0.470603
emergency_fund           -0.656365
credit_score             -1.097378
dtype: float64


In [11]:
for col in numerical_cols:
    Q1 = dfc[col].quantile(0.25)
    Q3 = dfc[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = dfc[(dfc[col] < lower) | (dfc[col] > upper)][col].count()
    print(f"{col} — Outliers: {outliers} ({round(outliers/len(dfc)*100, 2)}%)")

age — Outliers: 0 (0.0%)
gender — Outliers: 0 (0.0%)
marital_status — Outliers: 96963 (23.95%)
education — Outliers: 0 (0.0%)
monthly_salary — Outliers: 12120 (2.99%)
employment_type — Outliers: 121701 (30.06%)
years_of_employment — Outliers: 0 (0.0%)
company_type — Outliers: 0 (0.0%)
house_type — Outliers: 0 (0.0%)
monthly_rent — Outliers: 0 (0.0%)
family_size — Outliers: 0 (0.0%)
dependents — Outliers: 0 (0.0%)
school_fees — Outliers: 0 (0.0%)
college_fees — Outliers: 0 (0.0%)
travel_expenses — Outliers: 1360 (0.34%)
groceries_utilities — Outliers: 936 (0.23%)
other_monthly_expenses — Outliers: 953 (0.24%)
existing_loans — Outliers: 0 (0.0%)
current_emi_amount — Outliers: 0 (0.0%)
credit_score — Outliers: 5840 (1.44%)
bank_balance — Outliers: 13372 (3.3%)
emergency_fund — Outliers: 6090 (1.5%)
emi_scenario — Outliers: 0 (0.0%)
requested_amount — Outliers: 2298 (0.57%)
requested_tenure — Outliers: 7730 (1.91%)
emi_eligibility — Outliers: 91932 (22.71%)
max_monthly_emi — Outliers: 0 (0

In [24]:
skewed_cols = ['savings_ratio','monthly_salary','salary_credit_interaction','emi_affordability','expense_to_income','loan_income_ratio',
              'emi_burden', 'debt_to_income', 'financial_stability', 'emi_eligibility' , 'bank_balance', 'marital_status']

for col in skewed_cols:
    dfc[col] = np.log1p(dfc[col])

print("Log Transform done!")

Log Transform done!


In [28]:
log_cols = [
    "monthly_salary",
    "college_fees",
    "school_fees",
    "bank_balance",
    "emergency_fund",
    "requested_amount",
    "max_monthly_emi"
]

import numpy as np

for col in log_cols:
    dfc[col] = np.log1p(dfc[col])  # safe log

In [43]:
print(dfc[numerical_cols].skew().sort_values(ascending=False))

emi_eligibility           1.392009
marital_status            1.220566
college_fees              1.032556
requested_tenure          0.918680
age                       0.652411
current_emi_amount        0.415194
existing_loans            0.411897
monthly_rent              0.378475
education                 0.298449
house_type                0.091839
family_size               0.022211
dependents                0.022211
emi_scenario             -0.000145
employment_type          -0.075761
school_fees              -0.152274
groceries_utilities      -0.181008
travel_expenses          -0.183910
other_monthly_expenses   -0.213300
monthly_salary           -0.265037
max_monthly_emi          -0.352008
company_type             -0.408423
gender                   -0.408985
requested_amount         -0.467136
years_of_employment      -0.470603
bank_balance             -0.695439
emergency_fund           -0.720374
credit_score             -1.097378
dtype: float64


In [29]:
print(dfc[numerical_cols].skew().sort_values(ascending=False))

emi_eligibility           1.392009
marital_status            1.220566
college_fees              1.032556
requested_tenure          0.918680
age                       0.652411
current_emi_amount        0.415194
existing_loans            0.411897
monthly_rent              0.378475
education                 0.298449
house_type                0.091839
family_size               0.022211
dependents                0.022211
emi_scenario             -0.000145
employment_type          -0.075761
school_fees              -0.152274
groceries_utilities      -0.181008
travel_expenses          -0.183910
other_monthly_expenses   -0.213300
monthly_salary           -0.265037
max_monthly_emi          -0.352008
company_type             -0.408423
gender                   -0.408985
requested_amount         -0.467136
years_of_employment      -0.470603
bank_balance             -0.695439
emergency_fund           -0.720374
credit_score             -1.097378
dtype: float64


In [32]:
# Define features and targets
# Regression target
X = dfc.drop(columns=['max_monthly_emi', 'emi_eligibility'])
y_regression = dfc['max_monthly_emi']

# Classification target
y_classification = dfc['emi_eligibility']

print(f"X shape: {X.shape}")
print(f"y regression shape: {y_regression.shape}")
print(f"y classification shape: {y_classification.shape}")
print(f"\nClass distribution:\n{y_classification.value_counts()}")

X shape: (404800, 36)
y regression shape: (404800,)
y classification shape: (404800,)

Class distribution:
emi_eligibility
0    312868
2     74444
1     17488
Name: count, dtype: int64


In [34]:
dfc['emi_eligibility'].unique()

array([0, 2, 1])

In [35]:
from sklearn.model_selection import train_test_split

X = dfc.drop(columns=['max_monthly_emi', 'emi_eligibility'])
y_regression = dfc['max_monthly_emi']
y_classification = dfc['emi_eligibility']

# Regression split
X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X, y_regression, test_size=0.2, random_state=42)

# Classification split — stratified
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_classification, test_size=0.2, random_state=42, stratify=y_classification)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"X_train_cls: {X_train_cls.shape}, X_test_cls: {X_test_cls.shape}")

X_train: (323840, 36), X_test: (80960, 36)
X_train_cls: (323840, 36), X_test_cls: (80960, 36)


In [36]:
from sklearn.model_selection import StratifiedKFold, KFold

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("KFold and StratifiedKFold defined!")

KFold and StratifiedKFold defined!


In [ ]:
dfc.to_csv(r"D:\Emi Predict\Data Cleaning\emi_prediction_cleanafterFE.csv",header='infer',index=False)